In [19]:
import pandas as pd
import numpy as np
import requests
import pytz
import datetime as dt
from dotenv import load_dotenv
import os

from sklearn.linear_model import LinearRegression
import joblib

In [20]:
serial_numbers = [
    'MOD-00589',
    'MOD-00590',
    'MOD-00591',
    'MOD-00592',
    'MOD-00593',
    'MOD-00595',
]

In [21]:
# test the model on the training data
def preprocess(file_path):
    sensor = pd.read_csv(file_path, parse_dates=['timestamp'])
    sensor = sensor[['timestamp_local','rh','temp','pm25']].dropna(subset=['pm25'])
    sensor.rename(columns={'timestamp_local':'time'}, inplace=True)
    
    sensor['time'] = pd.to_datetime(sensor['time'])
    est = pytz.timezone('US/Eastern')
    sensor['time'] = sensor['time'].dt.tz_convert(est)
    sensor['time'] = sensor['time'].dt.strftime('%Y-%m-%d %H:%M:%S')
    sensor['time'] = pd.to_datetime(sensor['time'])
    sensor['day'] = sensor['time'].dt.date
    sensor['dayhour'] = sensor['time'].dt.strftime('%Y-%m-%d %H')

    sensor = sensor[::-1]

    # test on last 1/4 of data
    # split_index = int(len(sensor) * 0.75)
    # sensor = sensor[split_index:]

    return sensor

In [22]:
def create_enhanced_features(df, hourly=True):
    features = df.copy()
    if hourly:
        # Original features
        features['pm25'] = df['shourlyPM25']
        features['rh'] = df['shourlyRH']
        features['temp'] = df['shourlyTemp']
        
        # Interaction terms - these capture how RH/temp affect bias differently at different PM levels
        features['pm25_x_rh'] = df['shourlyPM25'] * df['shourlyRH']
        features['pm25_x_temp'] = df['shourlyPM25'] * df['shourlyTemp']
        features['rh_x_temp'] = df['shourlyRH'] * df['shourlyTemp']
        
        # Polynomial terms - capture non-linear effects
        features['pm25_sq'] = df['shourlyPM25'] ** 2
        features['rh_sq'] = df['shourlyRH'] ** 2
        features['temp_sq'] = df['shourlyTemp'] ** 2
    else:
        # Original features
        features['pm25'] = df['sdailyPM25']
        features['rh'] = df['sdailyRH']
        features['temp'] = df['sdailyTemp']
        
        # Interaction terms - these capture how RH/temp affect bias differently at different PM levels
        features['pm25_x_rh'] = df['sdailyPM25'] * df['sdailyRH']
        features['pm25_x_temp'] = df['sdailyPM25'] * df['sdailyTemp']
        features['rh_x_temp'] = df['sdailyRH'] * df['sdailyTemp']
        
        # Polynomial terms - capture non-linear effects
        features['pm25_sq'] = df['sdailyPM25'] ** 2
        features['rh_sq'] = df['sdailyRH'] ** 2
        features['temp_sq'] = df['sdailyTemp'] ** 2
    
    return features

In [23]:
# calculate prediction using given model
def calculate_prediction(df, model, groupBy) -> pd.DataFrame:
    if groupBy == 'dayhour':
        temp = df.groupby(groupBy, as_index=False).agg(
            shourlyPM25=('pm25', lambda x: x.mean(skipna=True)),
            shourlyRH=('rh', lambda x: x.mean(skipna=True)),
            shourlyTemp=('temp', lambda x: x.mean(skipna=True))
        ).reset_index()
        
        # temp['corrected_PM25h'] = model.predict(temp[['shourlyPM25','shourlyRH','shourlyTemp']])
        # return temp[['dayhour', 'corrected_PM25h']]

        X = create_enhanced_features(temp)
        X_scaled = model['scaler'].transform(X[model['feature_cols']])
        temp['corrected_PM25h'] = model['model'].predict(X_scaled)
        return temp[['dayhour', 'corrected_PM25h']]
    else:
        temp = df.groupby(groupBy).agg(
            sdailyPM25=('pm25', lambda x: x.mean(skipna=True)),
            sdailyRH=('rh', lambda x: x.mean(skipna=True)),
            sdailyTemp=('temp', lambda x: x.mean(skipna=True))
        ).reset_index()
        
        # temp['corrected_PM25d'] = model.predict(temp[['sdailyPM25','sdailyRH','sdailyTemp']])
        # return temp[['day', 'corrected_PM25d']]

        X = create_enhanced_features(temp, hourly=False)
        X_scaled = model['scaler'].transform(X[model['feature_cols']])
        temp['corrected_PM25d'] = model['model'].predict(X_scaled)
        return temp[['day', 'corrected_PM25d']]

In [24]:
today = dt.datetime.today().strftime('%Y-%m-%d')
tempDay = (dt.datetime.today() - dt.timedelta(days=5)).strftime('%Y-%m-%d')

for sn in serial_numbers:
    # load model
    model_number = sn.split('-')[-1]
    hourly_model = joblib.load(f'models/hmodel_{model_number}.joblib')
    daily_model = joblib.load(f'models/dmodel_{model_number}.joblib')

    # get data
    data_fetch_url = f'https://api.quant-aq.com/device-api/v1/devices/{sn}/data-by-date/{tempDay}'
    formatted_raw_data = preprocess(f'../ShortenedData/MOD-{model_number}-RAW.csv')

    # format data

    # calculate prediction
    hourly_prediction = calculate_prediction(formatted_raw_data, hourly_model, 'dayhour')
    daily_prediction = calculate_prediction(formatted_raw_data, daily_model, 'day')

    hourly_prediction.to_csv(f'Predictions/hMOD-{model_number}-PRED.csv', index=False)
    daily_prediction.to_csv(f'Predictions/dMOD-{model_number}-PRED.csv', index=False)
    

In [25]:
hourly_model = joblib.load(f'models/hmodel_00589.joblib')
daily_model = joblib.load(f'models/dmodel_00589.joblib')

# get data
formatted_raw_data = preprocess(f'../presentationData/current-MOD-00589-RAW.csv')

# calculate prediction
hourly_prediction = calculate_prediction(formatted_raw_data, hourly_model, 'dayhour')
daily_prediction = calculate_prediction(formatted_raw_data, daily_model, 'day')
hourly_prediction.to_csv(f'../presentationData/hMOD-00589-PRED.csv', index=False)
daily_prediction.to_csv(f'../presentationData/dMOD-00589-PRED.csv', index=False)